# 01 - EDA: Z-Alizadeh Sani extension

303 patients x 59 columns (UCI ID 411, CC BY 4.0, raw file in `data/raw/`).
All heavy lifting lives in `backend/app/data.py` and `backend/app/eda.py` so this
notebook stays in sync with the pipeline. Run from `backend/` with:
`PYTHONPATH=$PWD uv run jupyter nbconvert --to notebook --execute ../notebooks/01_eda.ipynb`

In [ ]:
from app.data import load_raw, clean
from app.eda import class_balance, leakage_check

raw = load_raw()
print('raw shape:', raw.shape)
df, registry = clean(raw)
print('clean shape:', df.shape)
print('missing values:', int(df.isna().sum().sum()))

## Class balance
CAD is imbalanced (216 CAD / 87 Normal); vessels are moderately imbalanced.

In [ ]:
bal = class_balance(df)
print(bal.to_string(index=False))
ax = bal.plot.bar(x='target', y=['n_neg', 'n_pos'], stacked=True,
                  title='Class balance per target (n=303)')
ax.figure.tight_layout()

## Leakage check
Single-feature ROC-AUC of Cath/LAD/LCX/RCA against CAD. Cath is 1.0 by
definition (CAD = angiography result); vessels are ~0.76-0.90 because CAD is
defined as any-vessel stenosis. None of them may be used as features.

In [ ]:
leak = leakage_check(df)
print(leak.to_string(index=False))

## Distributions and correlations
See `docs/figures/eda_*.png` (regenerated headlessly via `uv run python -m app.eda`).

In [ ]:
num = df.select_dtypes(include='number').drop(columns=['cad', 'lad', 'lcx', 'rca', 'cath'])
print(num.describe().T[['min', 'max', 'mean']].to_string())
print('\nstrongest absolute correlations:')
c = num.corr().abs()
pairs = c.where(~__import__('numpy').eye(len(c), dtype=bool)).stack().sort_values(ascending=False)
print(pairs.head(8).to_string())